# Notebook 1: TNG50 simulation hdf5 raw data — tables of particles + 3D visualization
**Author:** Gustavo Fernandes Gonçalves
ORCID: [0009-0006-5887-6621](https://orcid.org/0009-0006-5887-6621)
Lattes: http://lattes.cnpq.br/7416443230735445

In [ ]:
import h5py
import pandas as pd
 
from tutorial_tools import summarize_fields, preview_particles
 
pd.set_option("display.max_colwidth", None)

## Parameters
Simulation data available at [tng-project.org/data](https://www.tng-project.org/data/). Field descriptions at the [specifications page](https://www.tng-project.org/data/docs/specifications/).

In [ ]:
#subhalo 10
RAW_CUTOUT_PATH = "./TNG50_hdf5_cutouts/cutout_TNG50-1_snap97_subhalo10.hdf5"
FIREFLY_HDF5_PATH = "./TNG50_hdf5_cutouts/subhalo10_firefly_positions.hdf5"

#subhalo 406122
#RAW_CUTOUT_PATH = "./TNG50_hdf5_cutouts/cutout_TNG50-1_snap98_subhalo406122.hdf5"
#FIREFLY_HDF5_PATH = "./TNG50_hdf5_cutouts/subhalo406122_firefly_positions.hdf5"



PORT = 5500
 
PARTTYPES = [("PartType0", "gas"), ("PartType4", "stars"), ("PartType1", "dm")]

## 1. Simulation header


In [ ]:
with h5py.File(RAW_CUTOUT_PATH, "r") as f:
    header = dict(f["Header"].attrs)
 
for key in ("Time", "Redshift", "HubbleParam", "BoxSize", "NumPart_ThisFile", "MassTable"):
    if key in header:
        print(f"{key}: {header[key]}")

## 2. What is inside the simulation cutout?


In [ ]:
for ptype_key, label in PARTTYPES:
    print(f"--- {label} ({ptype_key}) ---")
    display(summarize_fields(RAW_CUTOUT_PATH, ptype_key))

## 3. Particle sample


In [ ]:
N_PREVIEW = 5
 
for ptype_key, label in PARTTYPES:
    print(f"--- {label} ({ptype_key}) ---")
    display(preview_particles(RAW_CUTOUT_PATH, ptype_key, n=N_PREVIEW))

In [ ]:
PARTTYPES = [("PartType0", "gas"), ("PartType4", "stars"), ("PartType1", "dm")]
coords = {}
masses = {}

with h5py.File(RAW_CUTOUT_PATH, "r") as f:
    header = f["Header"].attrs
    mass_table = header["MassTable"]  # fixed mass per type (0 if variable)

    for ptype, name in PARTTYPES:
        coords[name] = f[ptype]["Coordinates"][:]
        ptype_num = int(ptype.replace("PartType", ""))

        if "Masses" in f[ptype]:
            masses[name] = f[ptype]["Masses"][:] * 1e10
        else:
            n_part = coords[name].shape[0]
            masses[name] = np.full(n_part, mass_table[ptype_num] * 1e10)

# --- center of mass coordinates (using DM) ---
x = coords["dm"][:, 0]
y = coords["dm"][:, 1]
z = coords["dm"][:, 2]
m = masses["dm"]

xcom = np.sum(m * x) / np.sum(m)
ycom = np.sum(m * y) / np.sum(m)
zcom = np.sum(m * z) / np.sum(m)

halo_center = np.array([xcom, ycom, zcom])
print("Halo center:", halo_center)

# --- recenter all particle types ---
for name in coords:
    coords[name] = coords[name] - halo_center

In [ ]:
import matplotlib.pyplot as plt

name = "dm"
xyz = coords[name]

frac = 0.01
n_total = xyz.shape[0]
n_sample = int(n_total * frac)
idx = np.random.choice(n_total, size=n_sample, replace=False)
xyz_sub = xyz[idx]

fig, axes = plt.subplots(2, 1, figsize=(6, 12))
axes[0].scatter(xyz_sub[:, 0], xyz_sub[:, 1], s=1, alpha=0.5, c='tab:gray')
axes[0].set_xlabel("x [ckpc/h]"); axes[0].set_ylabel("y [ckpc/h]")
axes[0].set_aspect("equal")
axes[0].set_xlim(-100, 100)
axes[0].set_ylim(-100, 100)

axes[1].scatter(xyz_sub[:, 1], xyz_sub[:, 2], s=1, alpha=0.5, c='tab:gray')
axes[1].set_xlabel("y [ckpc/h]"); axes[1].set_ylabel("z [ckpc/h]")
axes[1].set_aspect("equal")
axes[1].set_xlim(-100, 100)
axes[1].set_ylim(-100, 100)

plt.tight_layout()
plt.show()

In [ ]:
# --- plot gas particles only, x-y and y-z---
name = "gas"
xyz = coords[name]

frac = 0.1
n_total = xyz.shape[0]
n_sample = int(n_total * frac)
idx = np.random.choice(n_total, size=n_sample, replace=False)
xyz_sub = xyz[idx]

fig, axes = plt.subplots(2, 1, figsize=(6, 12))
axes[0].scatter(xyz_sub[:, 0], xyz_sub[:, 1], s=0.1, alpha=0.5, c='tab:blue')
axes[0].set_xlabel("x [ckpc/h]"); axes[0].set_ylabel("y [ckpc/h]")
axes[0].set_aspect("equal")
axes[0].set_xlim(-50, 50)
axes[0].set_ylim(-50, 50)

axes[1].scatter(xyz_sub[:, 1], xyz_sub[:, 2], s=0.1, alpha=0.5, c='tab:blue')
axes[1].set_xlabel("y [ckpc/h]"); axes[1].set_ylabel("z [ckpc/h]")
axes[1].set_aspect("equal")
axes[1].set_xlim(-50, 50)
axes[1].set_ylim(-50, 50)

plt.tight_layout()
plt.show()

In [ ]:
# --- plot stars partciles only, x-y and y-z---
name = "stars"
xyz = coords[name]

frac = 0.01
n_total = xyz.shape[0]
n_sample = int(n_total * frac)
idx = np.random.choice(n_total, size=n_sample, replace=False)
xyz_sub = xyz[idx]

fig, axes = plt.subplots(2, 1, figsize=(6, 12))
axes[0].scatter(xyz_sub[:, 0], xyz_sub[:, 1], s=0.1, alpha=0.5, c='tab:orange')
axes[0].set_xlabel("x [ckpc/h]"); axes[0].set_ylabel("y [ckpc/h]")
axes[0].set_aspect("equal")
axes[0].set_xlim(-20, 20)
axes[0].set_ylim(-20, 20)

axes[1].scatter(xyz_sub[:, 1], xyz_sub[:, 2], s=0.1, alpha=0.5, c='tab:orange')
axes[1].set_xlabel("y [ckpc/h]"); axes[1].set_ylabel("z [ckpc/h]")
axes[1].set_aspect("equal")
axes[1].set_xlim(-20, 20)
axes[1].set_ylim(-20, 20)

plt.tight_layout()
plt.show()

## 4. 3D visualization (Firefly)


In [ ]:
from firefly.data_reader import SimpleReader
from firefly.server import spawnFireflyServer, quitAllFireflyServers
from IPython.display import display, HTML

quitAllFireflyServers()
reader = SimpleReader(FIREFLY_HDF5_PATH, write_to_disk=True)
spawnFireflyServer(port=PORT)

Shut down the server:


In [ ]:
quitAllFireflyServers()